# Hurricane Melissa, Recovery, Bauxite And Encroachment Threats Panel Figure

This notebook creates a stacked threats figure for the Nature Communications paper results section.

- **Panel a** shows Hurricane Melissa wind thresholds, storm track, and early post-event NDVI response in benefit-providing NbS areas.
- **Panel b** shows months 5-6 recovery among benefit-providing NbS areas that were damaged in months 1-2.
- **Panel c** shows river-flood forest restoration benefit areas that overlap mapped bauxite reserves and whether those overlap areas are protected.
- **Panel d** shows encroachment-risk land cover around benefit-providing coastal-flood mangroves and river-flood forest restoration areas.

White areas inside Jamaica are not existing forest by default. They are areas not plotted as benefit-providing NbS in the panel: outside the river-flood restoration avoided-EAD footprint and outside mapped benefit-providing mangrove patches.

The maps use the shared Jamaica map furniture helpers, with the north arrow and scale bar placed in the standard upper-right area following `dphil_papers/agents.md`.


In [ ]:
from pathlib import Path
import sys

BASE = Path("/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers")
ROBYN_LIBRARY_PATH = BASE / "robyns_libraries"
if str(ROBYN_LIBRARY_PATH) not in sys.path:
    sys.path.append(str(ROBYN_LIBRARY_PATH))

import geopandas as gpd
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import Robyn_paper_2_defs
from IPython.display import display
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.lines import Line2D
from rasterio.features import rasterize
from rasterio.warp import Resampling, reproject

Robyn_paper_2_defs.set_nature_style()
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)


## Paths And Constants

In [ ]:
PAPER2 = BASE / "dphil_paper_2"
PAPER3 = BASE / "dphil_paper_3"
COMMON = BASE / "dphil_common_cross_cutting"

OUT_DIR = PAPER3 / "results" / "threats" / "combined_threats" / "hurricane_melissa_bauxite_panel_figure"
OUT_DIR.mkdir(parents=True, exist_ok=True)

JAMAICA_BOUNDARY_PATH = COMMON / "common_incoming_data" / "boundaries" / "jamaica.gpkg"
MANGROVE_PATCHES_PATH = PAPER3 / "inputs" / "forces_of_nature_mangroves" / "mangroves.shp"
MANGROVE_PATCH_TABLE_PATH = PAPER3 / "results" / "threats" / "hurricane_melissa_damage" / "mangrove_eads_hurricane_damage" / "mangrove_ead_hurricane_damage_patch_table.csv"
MANGROVE_RECOVERY_PATCH_SUMMARY_PATH = PAPER3 / "results" / "threats" / "hurricane_melissa_damage" / "recovery" / "mangrove_ndvi_recovery_months1_6" / "mangrove_recovery_patch_summary.csv"
RIVER_EAD_MIN_PATH = PAPER2 / "processed_data" / "nbs_river_catchment" / "damage_reduction" / "damage_reduction_min.tif"
RIVER_EAD_MAX_PATH = PAPER2 / "processed_data" / "nbs_river_catchment" / "damage_reduction" / "damage_reduction_max.tif"
NDVI_BEFORE_PATH = PAPER3 / "inputs" / "ndvi" / "HLS_masked_NDVI_2months_before_epsg3448_2025-08-21_to_2025-10-21.tif"
NDVI_AFTER_PATH = PAPER3 / "inputs" / "ndvi" / "HLS_masked_NDVI_2months_after_epsg3448_2025-10-29_to_2025-12-29.tif"
NDVI_MONTHS_5_6_PATH = PAPER3 / "inputs" / "ndvi" / "HLS_masked_NDVI_months5to6_after_epsg3448_2026-03-01_to_2026-04-29.tif"
STORM_TRACK_PATH = PAPER3 / "inputs" / "hurricane_melissa_track_noaa" / "al132025_best_track" / "AL132025_lin.shp"
WIND_SWATH_PATH = PAPER3 / "inputs" / "hurricane_melissa_track_noaa" / "al132025_best_track" / "AL132025_windswath.shp"
BAUXITE_RESERVES_PATH = COMMON / "common_incoming_data" / "bauxite" / "Bauxite areas.shp"
FOREST_RESERVES_PATH = COMMON / "common_incoming_data" / "protected_landcover" / "forest_reserves.shp"
PROTECTED_AREAS_PATH = COMMON / "common_incoming_data" / "protected_landcover" / "protected_areas.shp"
RIVER_ENCROACHMENT_BY_CLASS_PATH = PAPER3 / "results" / "threats" / "encroachment_risk" / "river_flood_restoration_encroachment" / "river_flood_restoration_ring_encroachment_by_class.csv"
RIVER_ENCROACHMENT_RING_METADATA_PATH = PAPER3 / "results" / "threats" / "encroachment_risk" / "river_flood_restoration_encroachment" / "river_flood_restoration_ring_metadata.csv"
MANGROVE_ENCROACHMENT_DIR = PAPER3 / "processed_data" / "threats" / "encroachment_risk"
MANGROVE_EDGE_10M_BY_CLASS_RANGE_PATH = MANGROVE_ENCROACHMENT_DIR / "fn_mangrove_edge_10m_min_max_scenario_encroachment_by_class_range.csv"
MANGROVE_100M_BY_CLASS_RANGE_PATH = MANGROVE_ENCROACHMENT_DIR / "fn_mangrove_min_max_scenario_encroachment_by_class_range_100m.csv"
MANGROVE_EDGE_10M_SUMMARY_RANGE_PATH = MANGROVE_ENCROACHMENT_DIR / "fn_mangrove_edge_10m_min_max_scenario_encroachment_summary_range.csv"
MANGROVE_100M_SUMMARY_RANGE_PATH = MANGROVE_ENCROACHMENT_DIR / "fn_mangrove_min_max_scenario_encroachment_summary_range_100m.csv"

J2USD = 1.0 / 150.0
MAP_CRS = "EPSG:3448"
FIGURE_DPI = 300
REL_BASELINE_MIN = 0.20
REL_DAMAGE_THRESHOLD = -0.10
REL_GREENING_THRESHOLD = 0.10
WIND_THRESHOLD_ORDER = [34.0, 50.0, 64.0]

for input_path in [
    JAMAICA_BOUNDARY_PATH,
    MANGROVE_PATCHES_PATH,
    MANGROVE_PATCH_TABLE_PATH,
    MANGROVE_RECOVERY_PATCH_SUMMARY_PATH,
    RIVER_EAD_MIN_PATH,
    RIVER_EAD_MAX_PATH,
    NDVI_BEFORE_PATH,
    NDVI_AFTER_PATH,
    NDVI_MONTHS_5_6_PATH,
    STORM_TRACK_PATH,
    WIND_SWATH_PATH,
    BAUXITE_RESERVES_PATH,
    FOREST_RESERVES_PATH,
    PROTECTED_AREAS_PATH,
    RIVER_ENCROACHMENT_BY_CLASS_PATH,
    RIVER_ENCROACHMENT_RING_METADATA_PATH,
    MANGROVE_EDGE_10M_BY_CLASS_RANGE_PATH,
    MANGROVE_100M_BY_CLASS_RANGE_PATH,
    MANGROVE_EDGE_10M_SUMMARY_RANGE_PATH,
    MANGROVE_100M_SUMMARY_RANGE_PATH,
]:
    if not input_path.exists():
        raise FileNotFoundError(input_path)

OUT_DIR


## Helper Functions

In [ ]:
def clean_geometries(geodataframe: gpd.GeoDataFrame, target_crs: str) -> gpd.GeoDataFrame:
    """Reproject, repair, and remove empty geometries."""
    clean_geodataframe = geodataframe.to_crs(target_crs)
    clean_geodataframe = clean_geodataframe[clean_geodataframe.geometry.notna()].copy()
    clean_geodataframe = clean_geodataframe[~clean_geodataframe.geometry.is_empty].copy()
    clean_geodataframe["geometry"] = clean_geodataframe.geometry.make_valid()
    clean_geodataframe = clean_geodataframe[~clean_geodataframe.geometry.is_empty].copy()
    return clean_geodataframe


def read_noaa_lonlat_layer(path: Path, target_crs: str) -> gpd.GeoDataFrame:
    """Read NOAA best-track layers as lon/lat and reproject to the map CRS."""
    noaa_geodataframe = gpd.read_file(path)
    noaa_geodataframe = noaa_geodataframe.set_crs("EPSG:4326", allow_override=True)
    return clean_geometries(noaa_geodataframe, target_crs)


def read_positive_ead_usd(path: Path, reference_profile: dict | None = None) -> tuple[np.ndarray, dict]:
    """Read avoided EAD raster, convert JMD to USD, and retain only positive pixels."""
    with rasterio.open(path) as source_raster:
        profile = source_raster.profile.copy()
        ead_array = source_raster.read(1).astype("float64") * J2USD

    ead_array[~np.isfinite(ead_array) | (ead_array <= 0)] = np.nan

    if reference_profile is not None:
        alignment_checks = {
            "crs": profile["crs"] == reference_profile["crs"],
            "transform": profile["transform"] == reference_profile["transform"],
            "height": profile["height"] == reference_profile["height"],
            "width": profile["width"] == reference_profile["width"],
        }
        if not all(alignment_checks.values()):
            raise ValueError(f"Raster alignment mismatch for {path}: {alignment_checks}")

    return ead_array, profile


def reproject_continuous_to_reference(path: Path, reference_profile: dict) -> np.ndarray:
    """Reproject a continuous raster to the river restoration-benefit grid."""
    destination = np.full(
        (reference_profile["height"], reference_profile["width"]),
        np.nan,
        dtype="float32",
    )
    with rasterio.open(path) as source_raster:
        reproject(
            source=rasterio.band(source_raster, 1),
            destination=destination,
            src_transform=source_raster.transform,
            src_crs=source_raster.crs,
            src_nodata=source_raster.nodata,
            dst_transform=reference_profile["transform"],
            dst_crs=reference_profile["crs"],
            dst_nodata=np.nan,
            resampling=Resampling.bilinear,
        )
    return destination


def rasterize_geometries(geodataframe: gpd.GeoDataFrame, shape: tuple[int, int], transform) -> np.ndarray:
    """Rasterize valid geometries to a boolean mask on the reference grid."""
    raster_shapes = [(geometry, 1) for geometry in geodataframe.geometry if geometry is not None and not geometry.is_empty]
    if not raster_shapes:
        raise ValueError("No geometries available for rasterization")
    return rasterize(
        raster_shapes,
        out_shape=shape,
        transform=transform,
        fill=0,
        dtype="uint8",
        all_touched=False,
    ).astype(bool)



def classify_recovery_fraction_pct(recovery_fraction_pct: float) -> int:
    """Classify recovery fraction percentage for mapping."""
    if not np.isfinite(recovery_fraction_pct):
        return 1
    if recovery_fraction_pct <= 0:
        return 2
    if recovery_fraction_pct < 50:
        return 3
    if recovery_fraction_pct < 100:
        return 4
    return 5


def add_standard_jamaica_map_furniture(axis: plt.Axes, boundary_geodataframe: gpd.GeoDataFrame) -> None:
    """Add standard upper-right Jamaica map furniture using shared helpers."""
    scale_bar_point = Robyn_paper_2_defs.add_scale_bar(
        axis,
        boundary_geodataframe,
        where="right-top",
        pad=0.07,
        length_km=20,
        max_frac=0.22,
        lw=0.5,
        tick_h_frac=0.010,
        fs_lab=5.5,
        fs_unit=5.5,
        unit_text="km",
    )
    if scale_bar_point is None:
        return

    center_data_x, center_data_y = scale_bar_point
    center_axes_x, center_axes_y = axis.transAxes.inverted().transform(
        axis.transData.transform((center_data_x, center_data_y))
    )
    Robyn_paper_2_defs.add_north_arrow_axes(
        axis,
        center_axes_x,
        center_axes_y,
        size_frac=0.060,
        gap_frac=0.030,
        shaft_w_frac=0.10,
        head_w_frac=0.32,
        head_h_frac=0.55,
        fs=5.8,
        lw=0.5,
    )


def set_jamaica_extent(axis: plt.Axes, boundary_geodataframe: gpd.GeoDataFrame) -> None:
    """Apply consistent Jamaica map extent with enough upper-right room for map furniture."""
    minimum_x, minimum_y, maximum_x, maximum_y = boundary_geodataframe.total_bounds
    map_width = maximum_x - minimum_x
    map_height = maximum_y - minimum_y
    axis.set_xlim(minimum_x - map_width * 0.025, maximum_x + map_width * 0.025)
    axis.set_ylim(minimum_y - map_height * 0.11, maximum_y + map_height * 0.10)
    axis.set_axis_off()


def save_panel_figure(figure: plt.Figure, stem: str) -> list[Path]:
    """Save the panel figure to PNG, PDF, and SVG."""
    output_paths = []
    for suffix in ["png", "pdf", "svg"]:
        output_path = OUT_DIR / f"{stem}.{suffix}"
        figure.savefig(output_path, bbox_inches="tight", facecolor="white", dpi=FIGURE_DPI)
        output_paths.append(output_path)
    return output_paths


## Load Shared Spatial Inputs

In [ ]:
river_ead_min_usd, river_profile = read_positive_ead_usd(RIVER_EAD_MIN_PATH)
river_ead_max_usd, _ = read_positive_ead_usd(RIVER_EAD_MAX_PATH, river_profile)
river_transform = river_profile["transform"]
river_shape = (river_profile["height"], river_profile["width"])
river_pixel_area_ha = abs(river_transform.a * river_transform.e) / 10_000
river_benefit_mask = np.isfinite(river_ead_min_usd) | np.isfinite(river_ead_max_usd)
river_left, river_bottom, river_right, river_top = rasterio.transform.array_bounds(
    river_profile["height"],
    river_profile["width"],
    river_transform,
)

jamaica_boundary = clean_geometries(gpd.read_file(JAMAICA_BOUNDARY_PATH), MAP_CRS)
mangrove_patches = clean_geometries(gpd.read_file(MANGROVE_PATCHES_PATH), MAP_CRS)
storm_track = read_noaa_lonlat_layer(STORM_TRACK_PATH, MAP_CRS)
wind_swath = read_noaa_lonlat_layer(WIND_SWATH_PATH, MAP_CRS)
wind_swath_by_threshold = wind_swath.dissolve(by="RADII", as_index=False)

mangrove_patch_table = pd.read_csv(MANGROVE_PATCH_TABLE_PATH)
mangrove_recovery_patch_summary = pd.read_csv(MANGROVE_RECOVERY_PATCH_SUMMARY_PATH)
mangrove_patches["Mangrove_ID"] = mangrove_patches["ID"].astype(int)
mangrove_benefit_patches = mangrove_patches.merge(
    mangrove_patch_table[
        [
            "Mangrove_ID",
            "positive_avoided_ead_either",
            "damaged_area_ha",
        ]
    ],
    on="Mangrove_ID",
    how="left",
)
mangrove_benefit_patches = mangrove_benefit_patches[
    mangrove_benefit_patches["positive_avoided_ead_either"].fillna(False).astype(bool)
].copy()
mangrove_benefit_patches = mangrove_benefit_patches.merge(
    mangrove_recovery_patch_summary[
        [
            "Mangrove_ID",
            "damaged_months_1_2_area_ha",
            "damaged_mean_recovery_fraction_of_initial_drop_by_months_5_6_pct",
        ]
    ],
    on="Mangrove_ID",
    how="left",
)
mangrove_benefit_patches["has_gt10pct_damage"] = mangrove_benefit_patches["damaged_area_ha"].fillna(0) > 0
mangrove_benefit_patches["recovery_class"] = 1
mangrove_recovery_damaged_mask = mangrove_benefit_patches["damaged_months_1_2_area_ha"].fillna(0) > 0
mangrove_benefit_patches.loc[mangrove_recovery_damaged_mask, "recovery_class"] = mangrove_benefit_patches.loc[
    mangrove_recovery_damaged_mask,
    "damaged_mean_recovery_fraction_of_initial_drop_by_months_5_6_pct",
].apply(classify_recovery_fraction_pct)

print(f"River-flood restoration benefit area: {river_benefit_mask.sum() * river_pixel_area_ha:,.1f} ha")
print(f"Benefit-providing mangrove patches: {len(mangrove_benefit_patches):,}")


## Prepare Panels A And B: Hurricane Melissa Damage And Recovery

In [ ]:
ndvi_before = reproject_continuous_to_reference(NDVI_BEFORE_PATH, river_profile)
ndvi_after = reproject_continuous_to_reference(NDVI_AFTER_PATH, river_profile)
ndvi_months_5_6 = reproject_continuous_to_reference(NDVI_MONTHS_5_6_PATH, river_profile)
ndvi_eligible_mask = (
    river_benefit_mask
    & np.isfinite(ndvi_before)
    & np.isfinite(ndvi_after)
    & (ndvi_before >= REL_BASELINE_MIN)
)

relative_ndvi_change = np.full(river_shape, np.nan, dtype="float32")
np.divide(
    ndvi_after - ndvi_before,
    ndvi_before,
    out=relative_ndvi_change,
    where=ndvi_eligible_mask,
)

river_damage_mask = ndvi_eligible_mask & (relative_ndvi_change <= REL_DAMAGE_THRESHOLD)
river_greening_mask = ndvi_eligible_mask & (relative_ndvi_change >= REL_GREENING_THRESHOLD)
river_other_mask = river_benefit_mask & ~(river_damage_mask | river_greening_mask)

hurricane_class = np.zeros(river_shape, dtype="uint8")
hurricane_class[river_other_mask] = 1
hurricane_class[river_damage_mask] = 2
hurricane_class[river_greening_mask] = 3
hurricane_class = np.ma.masked_where(hurricane_class == 0, hurricane_class)

hurricane_class_colors = {
    "other": "#d9d9d9",
    "decrease": "#d73027",
    "increase": "#1a9850",
}
hurricane_cmap = ListedColormap(
    [
        hurricane_class_colors["other"],
        hurricane_class_colors["decrease"],
        hurricane_class_colors["increase"],
    ]
)
hurricane_norm = BoundaryNorm([0.5, 1.5, 2.5, 3.5], hurricane_cmap.N)
wind_line_colors = {
    34.0: "#6bb6ff",
    50.0: "#2585d9",
    64.0: "#0057b8",
}

river_recovery_full_series_mask = ndvi_eligible_mask & np.isfinite(ndvi_months_5_6)
river_recovery_damage_mask = river_recovery_full_series_mask & (relative_ndvi_change <= REL_DAMAGE_THRESHOLD)
initial_ndvi_drop = ndvi_before - ndvi_after
river_recovery_fraction = np.full(river_shape, np.nan, dtype="float32")
river_recovery_valid_drop_mask = river_recovery_damage_mask & (initial_ndvi_drop > 0)
river_recovery_fraction[river_recovery_valid_drop_mask] = (
    ndvi_months_5_6[river_recovery_valid_drop_mask] - ndvi_after[river_recovery_valid_drop_mask]
) / initial_ndvi_drop[river_recovery_valid_drop_mask]

recovery_class_labels = {
    1: "Benefit area not in damaged recovery subset",
    2: "No improvement/further decline",
    3: "<50% of initial NDVI loss recovered",
    4: "50-<100% of initial NDVI loss recovered",
    5: "Recovered to/beyond pre-event NDVI",
}
recovery_class_colors = {
    1: "#d9d9d9",
    2: "#8c2d04",
    3: "#d73027",
    4: "#fdae61",
    5: "#1a9850",
}
recovery_class = np.zeros(river_shape, dtype="uint8")
recovery_class[river_benefit_mask & ~river_recovery_damage_mask] = 1
recovery_class[river_recovery_damage_mask & (river_recovery_fraction <= 0)] = 2
recovery_class[river_recovery_damage_mask & (river_recovery_fraction > 0) & (river_recovery_fraction < 0.5)] = 3
recovery_class[river_recovery_damage_mask & (river_recovery_fraction >= 0.5) & (river_recovery_fraction < 1.0)] = 4
recovery_class[river_recovery_damage_mask & (river_recovery_fraction >= 1.0)] = 5
recovery_class = np.ma.masked_where(recovery_class == 0, recovery_class)
recovery_cmap = ListedColormap([recovery_class_colors[class_value] for class_value in range(1, 6)])
recovery_norm = BoundaryNorm([0.5, 1.5, 2.5, 3.5, 4.5, 5.5], recovery_cmap.N)

hurricane_summary = pd.DataFrame([
    {
        "group": "river-flood restoration benefit pixels",
        "total_area_ha": river_benefit_mask.sum() * river_pixel_area_ha,
        "ndvi_eligible_area_ha": ndvi_eligible_mask.sum() * river_pixel_area_ha,
        "ndvi_decrease_gt10_area_ha": river_damage_mask.sum() * river_pixel_area_ha,
        "ndvi_increase_gt10_area_ha": river_greening_mask.sum() * river_pixel_area_ha,
    }
])

recovery_summary_rows = []
for class_value, class_label in recovery_class_labels.items():
    class_mask = np.asarray(recovery_class.filled(0) == class_value)
    recovery_summary_rows.append(
        {
            "ecosystem": "River forest restoration",
            "recovery_class": class_label,
            "pixel_count": int(class_mask.sum()),
            "patch_count": np.nan,
            "area_ha": class_mask.sum() * river_pixel_area_ha,
            "share_of_damaged_area_pct": (
                class_mask.sum() / river_recovery_damage_mask.sum() * 100
                if class_value > 1 and river_recovery_damage_mask.sum()
                else np.nan
            ),
        }
    )
    mangrove_class_subset = mangrove_benefit_patches[mangrove_benefit_patches["recovery_class"].eq(class_value)]
    mangrove_class_area_ha = mangrove_class_subset["damaged_months_1_2_area_ha"].fillna(0).sum()
    recovery_summary_rows.append(
        {
            "ecosystem": "Mangroves",
            "recovery_class": class_label,
            "pixel_count": np.nan,
            "patch_count": len(mangrove_class_subset),
            "area_ha": mangrove_class_area_ha,
            "share_of_damaged_area_pct": (
                mangrove_class_area_ha
                / mangrove_benefit_patches["damaged_months_1_2_area_ha"].fillna(0).sum()
                * 100
                if class_value > 1
                else np.nan
            ),
        }
    )
recovery_panel_summary = pd.DataFrame(recovery_summary_rows)

hurricane_summary


## Prepare Panel C: Bauxite Threats

In [ ]:
bauxite_reserves = clean_geometries(gpd.read_file(BAUXITE_RESERVES_PATH), MAP_CRS)
forest_reserves = clean_geometries(gpd.read_file(FOREST_RESERVES_PATH), MAP_CRS)
protected_declarations = clean_geometries(gpd.read_file(PROTECTED_AREAS_PATH), MAP_CRS)
protected_areas = gpd.GeoDataFrame(
    pd.concat(
        [forest_reserves[["geometry"]], protected_declarations[["geometry"]]],
        ignore_index=True,
    ),
    geometry="geometry",
    crs=MAP_CRS,
)

bauxite_mask = rasterize_geometries(bauxite_reserves, river_shape, river_transform)
protected_mask = rasterize_geometries(protected_areas, river_shape, river_transform)

bauxite_class = np.zeros(river_shape, dtype="uint8")
bauxite_class[river_benefit_mask & ~bauxite_mask] = 1
bauxite_class[river_benefit_mask & bauxite_mask & ~protected_mask] = 2
bauxite_class[river_benefit_mask & bauxite_mask & protected_mask] = 3
bauxite_class = np.ma.masked_where(bauxite_class == 0, bauxite_class)

bauxite_class_colors = {
    "not_bauxite": "#d8d8d8",
    "unprotected_bauxite": "#b76534",
    "protected_bauxite": "#1f8a70",
}
bauxite_cmap = ListedColormap(
    [
        bauxite_class_colors["not_bauxite"],
        bauxite_class_colors["unprotected_bauxite"],
        bauxite_class_colors["protected_bauxite"],
    ]
)
bauxite_norm = BoundaryNorm([0.5, 1.5, 2.5, 3.5], bauxite_cmap.N)

bauxite_summary = pd.DataFrame([
    {
        "group": "river-flood restoration benefit pixels",
        "total_area_ha": river_benefit_mask.sum() * river_pixel_area_ha,
        "on_bauxite_area_ha": (river_benefit_mask & bauxite_mask).sum() * river_pixel_area_ha,
        "on_bauxite_protected_area_ha": (river_benefit_mask & bauxite_mask & protected_mask).sum() * river_pixel_area_ha,
        "on_bauxite_unprotected_area_ha": (river_benefit_mask & bauxite_mask & ~protected_mask).sum() * river_pixel_area_ha,
    }
])
bauxite_summary


## Create Stacked Panel Figure

Panel d uses the encroachment-risk summaries for benefit-providing coastal-flood mangroves and river-flood forest restoration areas.

In [ ]:
river_encroachment_by_class = pd.read_csv(RIVER_ENCROACHMENT_BY_CLASS_PATH)
river_encroachment_ring_metadata = pd.read_csv(RIVER_ENCROACHMENT_RING_METADATA_PATH)
mangrove_edge_10m_by_class_range = pd.read_csv(MANGROVE_EDGE_10M_BY_CLASS_RANGE_PATH)
mangrove_100m_by_class_range = pd.read_csv(MANGROVE_100M_BY_CLASS_RANGE_PATH)
mangrove_edge_10m_summary_range = pd.read_csv(MANGROVE_EDGE_10M_SUMMARY_RANGE_PATH)
mangrove_100m_summary_range = pd.read_csv(MANGROVE_100M_SUMMARY_RANGE_PATH)

encroachment_residual_category = "Natural/semi-natural land cover and water"
encroachment_category_order = [
    "Buildings and other infrastructure",
    "Agriculture",
    "Bauxite extraction / quarry",
    "Plantation",
    encroachment_residual_category,
]
encroachment_category_colors = {
    "Buildings and other infrastructure": "#5f6368",
    "Agriculture": "#d8a53a",
    "Bauxite extraction / quarry": "#b76534",
    "Plantation": "#3f8f62",
    encroachment_residual_category: "#d9d9d9",
}
encroachment_service_order = [
    "Coastal-flood mangroves",
    "River-flood forest restoration",
]
encroachment_distance_order = [10, 100]


def append_encroachment_composition_rows(
    composition_rows: list[dict],
    service_label: str,
    service_short_label: str,
    distance_m: int,
    risk_pct_lookup: dict[str, float],
) -> None:
    """Append risk-class and residual composition rows for one service-distance pair."""
    total_risk_pct = sum(float(risk_pct_lookup.get(category, 0.0)) for category in encroachment_category_order[:-1])
    for category in encroachment_category_order[:-1]:
        composition_rows.append(
            {
                "service_label": service_label,
                "service_short_label": service_short_label,
                "distance_m": distance_m,
                "category": category,
                "pct_of_classified_surrounding_land_cover": float(risk_pct_lookup.get(category, 0.0)),
            }
        )
    composition_rows.append(
        {
            "service_label": service_label,
            "service_short_label": service_short_label,
            "distance_m": distance_m,
            "category": encroachment_residual_category,
            "pct_of_classified_surrounding_land_cover": max(0.0, 100.0 - total_risk_pct),
        }
    )


def extract_mangrove_metric(summary_range: pd.DataFrame, metric_name: str, scenario_column: str = "minimum_scenario_value") -> float:
    """Extract one avoided-EAD-positive metric from a mangrove min/max summary table."""
    metric_rows = summary_range[
        summary_range["analysis_group"].eq("avoided_ead_positive")
        & summary_range["metric"].eq(metric_name)
    ]
    if metric_rows.empty:
        return np.nan
    return float(metric_rows.iloc[0][scenario_column])


encroachment_composition_rows = []
for distance_m in encroachment_distance_order:
    distance_rows = river_encroachment_by_class[river_encroachment_by_class["distance_m"].eq(distance_m)]
    river_risk_pct_lookup = dict(zip(distance_rows["encroachment_class"], distance_rows["pct_of_classified_ring"]))
    append_encroachment_composition_rows(
        encroachment_composition_rows,
        "River-flood forest restoration",
        "River forest",
        distance_m,
        river_risk_pct_lookup,
    )

mangrove_encroachment_sources = [
    (10, mangrove_edge_10m_by_class_range, "pct_of_classified_edge_ring_patch_associated"),
    (100, mangrove_100m_by_class_range, "pct_of_classified_ring_patch_associated"),
]
for distance_m, by_class_range, metric_name in mangrove_encroachment_sources:
    metric_rows = by_class_range[
        by_class_range["analysis_group"].eq("avoided_ead_positive")
        & by_class_range["metric"].eq(metric_name)
    ]
    mangrove_risk_pct_lookup = dict(zip(metric_rows["encroachment_class"], metric_rows["minimum_scenario_value"]))
    append_encroachment_composition_rows(
        encroachment_composition_rows,
        "Coastal-flood mangroves",
        "Mangroves",
        distance_m,
        mangrove_risk_pct_lookup,
    )

encroachment_composition = pd.DataFrame(encroachment_composition_rows)
encroachment_composition["bar_label"] = (
    encroachment_composition["service_short_label"]
    + "\n"
    + encroachment_composition["distance_m"].astype(str)
    + " m"
)
encroachment_composition["service_order"] = encroachment_composition["service_label"].map(
    {service_label: index for index, service_label in enumerate(encroachment_service_order)}
)
encroachment_composition["distance_order"] = encroachment_composition["distance_m"].map(
    {distance_m: index for index, distance_m in enumerate(encroachment_distance_order)}
)
encroachment_composition = encroachment_composition.sort_values(
    ["service_order", "distance_order", "category"]
).reset_index(drop=True)

encroachment_panel_summary = encroachment_composition.pivot_table(
    index=["service_label", "service_short_label", "distance_m", "bar_label", "service_order", "distance_order"],
    columns="category",
    values="pct_of_classified_surrounding_land_cover",
    aggfunc="sum",
).reset_index()
encroachment_panel_summary.columns.name = None
encroachment_panel_summary["encroachment_risk_total_pct"] = encroachment_panel_summary[
    encroachment_category_order[:-1]
].sum(axis=1)

river_summary_rows = river_encroachment_ring_metadata.assign(
    service_label="River-flood forest restoration",
    source="river_flood_restoration_encroachment notebook",
).rename(
    columns={
        "classified_ring_area_ha": "classified_surrounding_area_ha",
        "encroachment_pct_of_classified_ring": "source_encroachment_risk_total_pct",
    }
)

mangrove_summary_rows = pd.DataFrame(
    [
        {
            "distance_m": 10,
            "service_label": "Coastal-flood mangroves",
            "source": "benefit-positive mangrove 10 m edge encroachment summary",
            "classified_surrounding_area_ha": extract_mangrove_metric(
                mangrove_edge_10m_summary_range,
                "classified_edge_ring_area_km2_patch_associated",
            )
            * 100,
            "encroachment_area_ha": extract_mangrove_metric(
                mangrove_edge_10m_summary_range,
                "edge_encroachment_area_km2_patch_associated",
            )
            * 100,
            "source_encroachment_risk_total_pct": extract_mangrove_metric(
                mangrove_edge_10m_summary_range,
                "edge_encroachment_pct_of_classified_ring_patch_associated",
            ),
            "benefit_area_ha": extract_mangrove_metric(mangrove_edge_10m_summary_range, "mangrove_area_ha"),
            "benefit_area_with_any_encroachment_ha": extract_mangrove_metric(
                mangrove_edge_10m_summary_range,
                "mangrove_area_ha_with_any_edge_encroachment",
            ),
            "pct_benefit_area_with_any_encroachment": extract_mangrove_metric(
                mangrove_edge_10m_summary_range,
                "pct_mangrove_area_with_any_edge_encroachment",
            ),
            "avoided_ead_usd_minimum": extract_mangrove_metric(mangrove_edge_10m_summary_range, "avoided_usd"),
            "avoided_ead_usd_maximum": extract_mangrove_metric(
                mangrove_edge_10m_summary_range,
                "avoided_usd",
                "maximum_scenario_value",
            ),
            "pct_avoided_ead_with_any_encroachment_minimum": extract_mangrove_metric(
                mangrove_edge_10m_summary_range,
                "pct_avoided_usd_with_any_edge_encroachment",
            ),
            "pct_avoided_ead_with_any_encroachment_maximum": extract_mangrove_metric(
                mangrove_edge_10m_summary_range,
                "pct_avoided_usd_with_any_edge_encroachment",
                "maximum_scenario_value",
            ),
        },
        {
            "distance_m": 100,
            "service_label": "Coastal-flood mangroves",
            "source": "benefit-positive mangrove 100 m encroachment summary",
            "classified_surrounding_area_ha": extract_mangrove_metric(
                mangrove_100m_summary_range,
                "classified_ring_area_km2_patch_associated",
            )
            * 100,
            "encroachment_area_ha": extract_mangrove_metric(
                mangrove_100m_summary_range,
                "encroachment_area_km2_patch_associated",
            )
            * 100,
            "source_encroachment_risk_total_pct": extract_mangrove_metric(
                mangrove_100m_summary_range,
                "encroachment_pct_of_classified_ring_patch_associated",
            ),
            "benefit_area_ha": extract_mangrove_metric(mangrove_100m_summary_range, "mangrove_area_ha"),
            "benefit_area_with_any_encroachment_ha": extract_mangrove_metric(
                mangrove_100m_summary_range,
                "mangrove_area_ha_with_any_100m_encroachment",
            ),
            "pct_benefit_area_with_any_encroachment": extract_mangrove_metric(
                mangrove_100m_summary_range,
                "pct_mangrove_area_with_any_100m_encroachment",
            ),
            "avoided_ead_usd_minimum": extract_mangrove_metric(mangrove_100m_summary_range, "avoided_usd"),
            "avoided_ead_usd_maximum": extract_mangrove_metric(
                mangrove_100m_summary_range,
                "avoided_usd",
                "maximum_scenario_value",
            ),
            "pct_avoided_ead_with_any_encroachment_minimum": extract_mangrove_metric(
                mangrove_100m_summary_range,
                "pct_avoided_usd_with_any_100m_encroachment",
            ),
            "pct_avoided_ead_with_any_encroachment_maximum": extract_mangrove_metric(
                mangrove_100m_summary_range,
                "pct_avoided_usd_with_any_100m_encroachment",
                "maximum_scenario_value",
            ),
        },
    ]
)

encroachment_service_summary = pd.concat(
    [river_summary_rows, mangrove_summary_rows],
    ignore_index=True,
    sort=False,
)
encroachment_panel_summary = encroachment_panel_summary.merge(
    encroachment_service_summary,
    on=["service_label", "distance_m"],
    how="left",
)
encroachment_panel_summary

In [ ]:
figure = plt.figure(figsize=(180 / 25.4, 292 / 25.4), dpi=FIGURE_DPI)
hurricane_axis = figure.add_axes([0.02, 0.815, 0.96, 0.130])
recovery_axis = figure.add_axes([0.02, 0.620, 0.96, 0.130])
bauxite_axis = figure.add_axes([0.02, 0.430, 0.96, 0.120])
encroachment_axis = figure.add_axes([0.170, 0.205, 0.700, 0.175])
panel_label_x = 0.145
figure.text(panel_label_x, 0.930, "a", fontsize=8, fontweight="bold", va="top", ha="left")
figure.text(panel_label_x, 0.735, "b", fontsize=8, fontweight="bold", va="top", ha="left")
figure.text(panel_label_x, 0.540, "c", fontsize=8, fontweight="bold", va="top", ha="left")
figure.text(panel_label_x, 0.385, "d", fontsize=8, fontweight="bold", va="top", ha="left")

# Panel a: Hurricane Melissa.
jamaica_boundary.plot(ax=hurricane_axis, facecolor="white", edgecolor="none", zorder=0)
hurricane_axis.imshow(
    hurricane_class,
    extent=(river_left, river_right, river_bottom, river_top),
    origin="upper",
    cmap=hurricane_cmap,
    norm=hurricane_norm,
    interpolation="nearest",
    zorder=2,
)
mangrove_benefit_patches.loc[~mangrove_benefit_patches["has_gt10pct_damage"]].plot(
    ax=hurricane_axis,
    facecolor="#9bd8d2",
    edgecolor="#006d77",
    linewidth=0.18,
    alpha=0.95,
    zorder=4,
)
mangrove_benefit_patches.loc[mangrove_benefit_patches["has_gt10pct_damage"]].plot(
    ax=hurricane_axis,
    facecolor="#08519c",
    edgecolor="#08306b",
    linewidth=0.18,
    alpha=0.95,
    zorder=5,
)
for wind_threshold in WIND_THRESHOLD_ORDER:
    wind_threshold_geodataframe = wind_swath_by_threshold[wind_swath_by_threshold["RADII"].astype(float).eq(wind_threshold)]
    if not wind_threshold_geodataframe.empty:
        wind_threshold_geodataframe.boundary.plot(
            ax=hurricane_axis,
            color=wind_line_colors[wind_threshold],
            linewidth=0.75 if wind_threshold < 64 else 0.95,
            zorder=7,
        )
storm_track.plot(ax=hurricane_axis, color="black", linewidth=0.85, zorder=8)
jamaica_boundary.boundary.plot(ax=hurricane_axis, color="black", linewidth=0.55, zorder=9)
set_jamaica_extent(hurricane_axis, jamaica_boundary)
add_standard_jamaica_map_furniture(hurricane_axis, jamaica_boundary)
hurricane_axis.set_title("Early NDVI response to Hurricane Melissa", pad=2)

hurricane_handles = [
    mpatches.Patch(facecolor="white", edgecolor="#bdbdbd", label="Not plotted as benefit-providing NbS"),
    mpatches.Patch(facecolor=hurricane_class_colors["decrease"], edgecolor="none", label="Forest restoration benefit: NDVI decrease >10%"),
    mpatches.Patch(facecolor=hurricane_class_colors["increase"], edgecolor="none", label="Forest restoration benefit: NDVI increase >10%"),
    mpatches.Patch(facecolor=hurricane_class_colors["other"], edgecolor="none", label="Forest restoration benefit: <10% NDVI change or not NDVI eligible"),
    mpatches.Patch(facecolor="#9bd8d2", edgecolor="#006d77", label="Mangrove benefit patch"),
    mpatches.Patch(facecolor="#08519c", edgecolor="#08306b", label="Damaged mangrove benefit patch"),
    Line2D([0], [0], color=wind_line_colors[34.0], linewidth=0.8, label="34 kt wind threshold"),
    Line2D([0], [0], color=wind_line_colors[50.0], linewidth=0.8, label="50 kt wind threshold"),
    Line2D([0], [0], color=wind_line_colors[64.0], linewidth=1.0, label="64 kt wind threshold"),
    Line2D([0], [0], color="black", linewidth=0.85, label="Melissa track"),
]
hurricane_axis.legend(
    handles=hurricane_handles,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.015),
    ncol=3,
    frameon=True,
    framealpha=1.0,
    facecolor="white",
    edgecolor="#cfcfcf",
    borderpad=0.40,
    handlelength=1.3,
    columnspacing=0.85,
)

# Panel b: months 5-6 recovery.
jamaica_boundary.plot(ax=recovery_axis, facecolor="white", edgecolor="none", zorder=0)
recovery_axis.imshow(
    recovery_class,
    extent=(river_left, river_right, river_bottom, river_top),
    origin="upper",
    cmap=recovery_cmap,
    norm=recovery_norm,
    interpolation="nearest",
    zorder=2,
)
for class_value in range(1, 6):
    patch_subset = mangrove_benefit_patches[mangrove_benefit_patches["recovery_class"].eq(class_value)]
    if not patch_subset.empty:
        patch_subset.plot(
            ax=recovery_axis,
            facecolor=recovery_class_colors[class_value],
            edgecolor="#08306b" if class_value > 1 else "#7f7f7f",
            linewidth=0.20 if class_value > 1 else 0.08,
            alpha=0.95 if class_value > 1 else 0.65,
            zorder=5 if class_value > 1 else 4,
        )
for wind_threshold in WIND_THRESHOLD_ORDER:
    wind_threshold_geodataframe = wind_swath_by_threshold[wind_swath_by_threshold["RADII"].astype(float).eq(wind_threshold)]
    if not wind_threshold_geodataframe.empty:
        wind_threshold_geodataframe.boundary.plot(
            ax=recovery_axis,
            color=wind_line_colors[wind_threshold],
            linewidth=0.75 if wind_threshold < 64 else 0.95,
            zorder=7,
        )
storm_track.plot(ax=recovery_axis, color="black", linewidth=0.85, zorder=8)
jamaica_boundary.boundary.plot(ax=recovery_axis, color="black", linewidth=0.55, zorder=9)
set_jamaica_extent(recovery_axis, jamaica_boundary)
add_standard_jamaica_map_furniture(recovery_axis, jamaica_boundary)
recovery_axis.set_title("Months 5-6 NDVI recovery", pad=2)

recovery_handles = [
    mpatches.Patch(facecolor=recovery_class_colors[class_value], edgecolor="none", label=recovery_class_labels[class_value])
    for class_value in range(1, 6)
]
recovery_handles.extend(
    [
        Line2D([0], [0], color=wind_line_colors[34.0], linewidth=0.8, label="34 kt wind threshold"),
        Line2D([0], [0], color=wind_line_colors[50.0], linewidth=0.8, label="50 kt wind threshold"),
        Line2D([0], [0], color=wind_line_colors[64.0], linewidth=1.0, label="64 kt wind threshold"),
        Line2D([0], [0], color="black", linewidth=0.85, label="Melissa track"),
    ]
)
recovery_axis.legend(
    handles=recovery_handles,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.015),
    ncol=3,
    frameon=True,
    framealpha=1.0,
    facecolor="white",
    edgecolor="#cfcfcf",
    borderpad=0.40,
    handlelength=1.3,
    columnspacing=0.85,
)

# Panel c: Bauxite.
jamaica_boundary.plot(ax=bauxite_axis, facecolor="white", edgecolor="none", zorder=0)
bauxite_axis.imshow(
    bauxite_class,
    extent=(river_left, river_right, river_bottom, river_top),
    origin="upper",
    cmap=bauxite_cmap,
    norm=bauxite_norm,
    interpolation="nearest",
    zorder=2,
)
bauxite_reserves.boundary.plot(ax=bauxite_axis, color="#7a3d1f", linewidth=0.45, alpha=0.85, zorder=4)
jamaica_boundary.boundary.plot(ax=bauxite_axis, color="black", linewidth=0.55, zorder=5)
set_jamaica_extent(bauxite_axis, jamaica_boundary)
add_standard_jamaica_map_furniture(bauxite_axis, jamaica_boundary)
bauxite_axis.set_title("Bauxite reserves and restoration benefits", pad=2)

bauxite_handles = [
    mpatches.Patch(facecolor="white", edgecolor="#bdbdbd", label="Not river-flood restoration benefit area"),
    mpatches.Patch(facecolor=bauxite_class_colors["not_bauxite"], edgecolor="none", label="Restoration benefit area not on bauxite"),
    mpatches.Patch(facecolor=bauxite_class_colors["unprotected_bauxite"], edgecolor="none", label="Restoration benefit area on bauxite, unprotected"),
    mpatches.Patch(facecolor=bauxite_class_colors["protected_bauxite"], edgecolor="none", label="Restoration benefit area on bauxite, protected"),
    Line2D([0], [0], color="#7a3d1f", linewidth=0.8, label="Bauxite reserve boundary"),
]
bauxite_axis.legend(
    handles=bauxite_handles,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.015),
    ncol=3,
    frameon=True,
    framealpha=1.0,
    facecolor="white",
    edgecolor="#cfcfcf",
    borderpad=0.40,
    handlelength=1.3,
    columnspacing=0.85,
)

# Panel d: Encroachment-risk land-cover composition.
encroachment_bar_lookup = encroachment_panel_summary.sort_values(["service_order", "distance_order"])
bar_labels = encroachment_bar_lookup["bar_label"].tolist()
bar_y_positions = np.arange(len(bar_labels))[::-1]
bar_left = np.zeros(len(bar_labels))
for category in encroachment_category_order:
    category_values = [
        encroachment_composition.loc[
            (encroachment_composition["bar_label"].eq(bar_label))
            & (encroachment_composition["category"].eq(category)),
            "pct_of_classified_surrounding_land_cover",
        ].sum()
        for bar_label in bar_labels
    ]
    encroachment_axis.barh(
        bar_y_positions,
        category_values,
        left=bar_left,
        color=encroachment_category_colors[category],
        edgecolor="white",
        linewidth=0.35,
        height=0.68,
        label=category,
    )
    bar_left += np.array(category_values)

encroachment_axis.set_xlim(0, 100)
encroachment_axis.set_yticks(bar_y_positions)
encroachment_axis.set_yticklabels(bar_labels)
encroachment_axis.set_xlabel("Share of classified surrounding land cover (%)", fontsize=6.5, labelpad=2)
encroachment_axis.set_title("Encroachment-risk land cover around benefit-providing NbS", pad=2)
encroachment_axis.tick_params(axis="both", labelsize=6)
encroachment_axis.grid(axis="x", linewidth=0.30, alpha=0.35)
encroachment_axis.spines[["top", "right"]].set_visible(False)
encroachment_axis.legend(
    loc="upper center",
    bbox_to_anchor=(0.5, -0.30),
    ncol=2,
    frameon=False,
    fontsize=5.8,
    columnspacing=1.1,
    handlelength=1.2,
)

panel_figure_paths = save_panel_figure(figure, "hurricane_melissa_bauxite_recovery_threats_panel_figure")
display(figure)
plt.close(figure)

panel_figure_paths


## Export Figure Metadata

In [ ]:
metadata_path = OUT_DIR / "hurricane_melissa_bauxite_recovery_threats_panel_figure_metadata.csv"
hurricane_summary_path = OUT_DIR / "hurricane_melissa_panel_summary.csv"
recovery_summary_path = OUT_DIR / "recovery_panel_summary.csv"
bauxite_summary_path = OUT_DIR / "bauxite_panel_summary.csv"
encroachment_summary_path = OUT_DIR / "encroachment_panel_summary.csv"
encroachment_composition_path = OUT_DIR / "encroachment_panel_composition.csv"

figure_metadata = pd.DataFrame([
    {"name": "jamaica_boundary_path", "value": str(JAMAICA_BOUNDARY_PATH)},
    {"name": "mangrove_patches_path", "value": str(MANGROVE_PATCHES_PATH)},
    {"name": "mangrove_patch_table_path", "value": str(MANGROVE_PATCH_TABLE_PATH)},
    {"name": "mangrove_recovery_patch_summary_path", "value": str(MANGROVE_RECOVERY_PATCH_SUMMARY_PATH)},
    {"name": "river_ead_min_path", "value": str(RIVER_EAD_MIN_PATH)},
    {"name": "river_ead_max_path", "value": str(RIVER_EAD_MAX_PATH)},
    {"name": "ndvi_before_path", "value": str(NDVI_BEFORE_PATH)},
    {"name": "ndvi_after_path", "value": str(NDVI_AFTER_PATH)},
    {"name": "ndvi_months_5_6_path", "value": str(NDVI_MONTHS_5_6_PATH)},
    {"name": "storm_track_path", "value": str(STORM_TRACK_PATH)},
    {"name": "wind_swath_path", "value": str(WIND_SWATH_PATH)},
    {"name": "bauxite_reserves_path", "value": str(BAUXITE_RESERVES_PATH)},
    {"name": "forest_reserves_path", "value": str(FOREST_RESERVES_PATH)},
    {"name": "protected_areas_path", "value": str(PROTECTED_AREAS_PATH)},
    {"name": "river_encroachment_by_class_path", "value": str(RIVER_ENCROACHMENT_BY_CLASS_PATH)},
    {"name": "river_encroachment_ring_metadata_path", "value": str(RIVER_ENCROACHMENT_RING_METADATA_PATH)},
    {"name": "mangrove_edge_10m_by_class_range_path", "value": str(MANGROVE_EDGE_10M_BY_CLASS_RANGE_PATH)},
    {"name": "mangrove_100m_by_class_range_path", "value": str(MANGROVE_100M_BY_CLASS_RANGE_PATH)},
    {"name": "mangrove_edge_10m_summary_range_path", "value": str(MANGROVE_EDGE_10M_SUMMARY_RANGE_PATH)},
    {"name": "mangrove_100m_summary_range_path", "value": str(MANGROVE_100M_SUMMARY_RANGE_PATH)},
    {"name": "map_crs", "value": MAP_CRS},
    {"name": "river_benefit_area_definition", "value": "positive avoided EAD in minimum or maximum river-flood restoration scenario"},
    {"name": "recovery_panel_definition", "value": "months 5-6 recovery fraction among pixels/patches damaged in months 1-2"},
    {"name": "white_background_definition", "value": "areas not plotted as benefit-providing NbS in the panel, not necessarily existing forest"},
    {"name": "north_arrow_and_scale_bar", "value": "shared Robyn_paper_2_defs helpers, right-top placement"},
])

figure_metadata.to_csv(metadata_path, index=False)
hurricane_summary.to_csv(hurricane_summary_path, index=False)
recovery_panel_summary.to_csv(recovery_summary_path, index=False)
bauxite_summary.to_csv(bauxite_summary_path, index=False)
encroachment_panel_summary.to_csv(encroachment_summary_path, index=False)
encroachment_composition.to_csv(encroachment_composition_path, index=False)

[
    metadata_path,
    hurricane_summary_path,
    recovery_summary_path,
    bauxite_summary_path,
    encroachment_summary_path,
    encroachment_composition_path,
]
